# TEB Demo (Timed Elastic Band)

This notebook is a self-learning walkthrough of a simplified TEB-style local trajectory optimization.

You will optimize an initial trajectory by balancing smoothness, obstacle clearance, and goal consistency.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

## 1. Set local planning scene

In [ ]:
start = np.array([0.8, 0.8])
goal = np.array([10.0, 9.2])

obstacles = np.array([
    [3.8, 4.2], [4.4, 4.6], [5.0, 4.9],
    [6.2, 6.4], [6.8, 6.7], [7.3, 6.6]
])

xmin, xmax = 0.0, 11.0
ymin, ymax = 0.0, 10.5

## 2. Initialize an elastic-band trajectory

We discretize the trajectory into `N` points between start and goal.

In [ ]:
N = 22
alpha = np.linspace(0.0, 1.0, N)[:, None]
traj = (1 - alpha) * start + alpha * goal

# Time intervals between points (simplified fixed initial guess)
dt = np.full(N - 1, 0.2)

## 3. Define simplified TEB objective terms

Main terms in this educational variant:
- smoothness term (second-difference penalty),
- obstacle repulsion term (keep points away from obstacles),
- progress/consistency term (keep endpoints anchored).

In [ ]:
w_smooth = 1.6
w_obs = 2.8
w_anchor = 8.0
safe_dist = 0.9

def smoothness_grad(path):
    g = np.zeros_like(path)
    for i in range(1, len(path) - 1):
        g[i] += 2 * path[i] - path[i - 1] - path[i + 1]
    return g

def obstacle_grad(path, obs, d_safe):
    g = np.zeros_like(path)
    for i in range(1, len(path) - 1):
        p = path[i]
        for o in obs:
            vec = p - o
            d = np.linalg.norm(vec) + 1e-6
            if d < d_safe:
                # Push away stronger when closer than safe distance
                g[i] += (d_safe - d) * (vec / d)
    return g

def objective(path, obs):
    smooth = 0.0
    for i in range(1, len(path) - 1):
        v = path[i - 1] - 2 * path[i] + path[i + 1]
        smooth += np.dot(v, v)

    obs_cost = 0.0
    for i in range(1, len(path) - 1):
        p = path[i]
        dmin = np.min(np.linalg.norm(obs - p, axis=1))
        if dmin < safe_dist:
            obs_cost += (safe_dist - dmin) ** 2

    anchor = np.sum((path[0] - start) ** 2) + np.sum((path[-1] - goal) ** 2)
    return w_smooth * smooth + w_obs * obs_cost + w_anchor * anchor

## 4. Optimize the elastic band

We run gradient-style updates and keep endpoints fixed at start and goal.

In [ ]:
traj0 = traj.copy()
history = []
lr = 0.08
iters = 120

for _ in range(iters):
    g_s = smoothness_grad(traj)
    g_o = obstacle_grad(traj, obstacles, safe_dist)

    grad = w_smooth * g_s - w_obs * g_o

    # Keep endpoints anchored
    grad[0] = 0.0
    grad[-1] = 0.0

    traj = traj - lr * grad
    traj[:, 0] = np.clip(traj[:, 0], xmin, xmax)
    traj[:, 1] = np.clip(traj[:, 1], ymin, ymax)

    traj[0] = start
    traj[-1] = goal

    history.append(objective(traj, obstacles))

print('Initial objective:', round(objective(traj0, obstacles), 3))
print('Final objective:', round(objective(traj, obstacles), 3))

## 5. Visualize trajectory before vs after optimization

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='black', s=40, label='obstacles')

ax.plot(traj0[:, 0], traj0[:, 1], '--', color='gray', linewidth=1.8, label='initial band')
ax.plot(traj[:, 0], traj[:, 1], color='deepskyblue', linewidth=2.6, label='optimized band')

ax.scatter(start[0], start[1], c='limegreen', s=90, marker='o', label='start')
ax.scatter(goal[0], goal[1], c='crimson', s=90, marker='*', label='goal')

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
ax.set_title('Simplified TEB trajectory optimization')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(history, color='slateblue')
plt.title('Objective over iterations')
plt.xlabel('iteration')
plt.ylabel('objective')
plt.tight_layout()
plt.show()

## 6. Self-study experiments

Try these exercises:
1. Increase `w_obs` and inspect extra obstacle clearance.
2. Increase `w_smooth` and inspect path curvature reduction.
3. Move obstacles closer to the straight line and compare optimization behavior.